### Find neighbours of atoms in PDB structures 

The PDB structure contains the (x, y, z) coordinates of every atom in the structure. We will now use the implementation of the  algorithm, `NeighborSearch`, from biopython that can find neighbouring atoms that are within a certain distance of a given coordinate.

In [2]:
import os
from Bio.PDB import PDBParser, NeighborSearch

path = os.path.join("data", "output.txt")
if os.path.exists(path):
    print("File available:", path)

# this is the directory where our PDB files that we want to use are stored. it will here be called PDB_DIR and a a small reminder rn we are still working with only the corona files 

PDB_DIR = "../data/pdbs_corona_cnf"

# now we neeed to find the path to the corresponding PDB File 

pdb_id = "6xc3"
filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")


first we will parse the PDB file

In [3]:
from Bio.PDB import PDBParser

parser = PDBParser(PERMISSIVE=1)
structure = parser.get_structure(pdb_id, filename)

Then we need to initialise the data structure that is used to perform the search of neighbours. Here we use all atoms of the structure for demonstration. This can be wasteful. Later, it can be advantageous to only use the atoms of the antigen, for example.

In [8]:
atoms = list(structure.get_atoms())
ns = NeighborSearch(atoms)
ns

Now we can use the `ns.search(coords, distance)` method to find all atoms that are within `distance` angstroms from the position specified by the `[x, y, z]` array `coords`.

We can retrieve the position of an atom using its `coord` property:

first we want to see how big our total structure is

In [11]:
from Bio.PDB import PDBParser
import numpy as np

# Load structure
parser = PDBParser()
structure = parser.get_structure(pdb_id, filename)

# Collect all atom coordinates
coords = []
for model in structure:
    for chain in model:
        for residue in chain:
            for atom in residue:
                coords.append(atom.coord)

# Convert to NumPy array
coords = np.array(coords)

# Compute bounding box
min_coords = coords.min(axis=0)
max_coords = coords.max(axis=0)
size = max_coords - min_coords

print("Min coords (x, y, z):", min_coords)
print("Max coords (x, y, z):", max_coords)
print("Size (x, y, z):", size)
print("Total structure span:", np.linalg.norm(size))

Min coords (x, y, z): [-102.57    21.256  -26.405]
Max coords (x, y, z): [ 24.164 116.249  57.886]
Size (x, y, z): [126.734  94.993  84.291]
Total structure span: 179.41614


In [5]:
atom = structure[0]['H'][52]['N']
atom.coord

array([-60.134,  62.357,  23.903], dtype=float32)

To find all atoms in vicinity of an atom we simply pass the coordinates of the atom to the `ns.search` method.

In [6]:
close_atoms = ns.search(atom.coord, 4.0)
close_atoms

[<Atom N>,
 <Atom O>,
 <Atom CD>,
 <Atom CE3>,
 <Atom CG>,
 <Atom CB>,
 <Atom CA>,
 <Atom C>,
 <Atom CG2>,
 <Atom N>,
 <Atom CA>,
 <Atom CB>,
 <Atom CG1>,
 <Atom O>,
 <Atom N>,
 <Atom C>,
 <Atom O>]

Now we want to find the residues and chains that theese atoms belong to

In [9]:
import pandas as pd

data = []

for atom in close_atoms:
    s, m, c, r, a = atom.full_id
    resnum = str(r[1]) + r[2]

    # to get the resname we need to access the residue the atom belongs to
    residue = atom.get_parent()
    resname = residue.get_resname()
    data.append(dict(chain = c, resnum = resnum, resname = resname, atom = a[0], Ab_Ag = filename))


df = pd.DataFrame(data)

df

,chain,resnum,resname,atom,Ab_Ag
0,H,51,ILE,N,../data/pdbs_corona_cnf/6xc3.pdb
1,H,51,ILE,O,../data/pdbs_corona_cnf/6xc3.pdb
2,H,52A,PRO,CD,../data/pdbs_corona_cnf/6xc3.pdb
3,H,33,TRP,CE3,../data/pdbs_corona_cnf/6xc3.pdb
4,H,52,TYR,CG,../data/pdbs_corona_cnf/6xc3.pdb
5,H,52,TYR,CB,../data/pdbs_corona_cnf/6xc3.pdb
6,H,52,TYR,CA,../data/pdbs_corona_cnf/6xc3.pdb
7,H,51,ILE,C,../data/pdbs_corona_cnf/6xc3.pdb
8,H,51,ILE,CG2,../data/pdbs_corona_cnf/6xc3.pdb
9,H,52,TYR,N,../data/pdbs_corona_cnf/6xc3.pdb


I now somehow want to create a Dataframe where all information on neighbouring atoms can be saved to for all 
here is a code that 
1. Loops through all atoms in your structure (e.g., the antibody).

2. Uses a NeighborSearch object built from another structure (e.g., the antigen).

3. Finds close atoms (within 4.0 Å) for each atom.

4. Keeps only those atoms that have neighbors in the antigen.

In [15]:
from Bio.PDB import PDBParser, NeighborSearch
from Bio.PDB import Selection

# --- Load both structures ---
parser = PDBParser()
structure_antibody = parser.get_structure(pdb_id, filename)
structure_antigen = parser.get_structure(pdb_id, filename)

# --- Get all atoms from the antigen for neighbor search ---
antigen_atoms = list(structure_antigen.get_atoms())
ns = NeighborSearch(antigen_atoms)

# --- Store close atoms from antibody ---
close_antibody_atoms = []

for model in structure_antibody:
    for chain in model:
        for residue in chain:
            for atom in residue:
                close_atoms = ns.search(atom.coord, 4.0)
                if close_atoms:  # If there are any nearby atoms
                    close_antibody_atoms.append(atom)

print(f"Found {len(close_antibody_atoms)} atoms in the antibody close to the antigen.")

Found 8333 atoms in the antibody close to the antigen.


i think at this point its useful to do some sensitivity testing 

`close_atoms` now contains the atom entities of all atoms that are within 4.0 angstrom of the supplied coordinates. To find the residues and chains these atoms belong to, we can either use repeated application of the `get_parent()` method, or we can simply use the `get_full_id()` method.

As we want the output as a DataFrame, we first create a vector of dicts, and then create the DataFrame. 